In [ ]:
import numpy as np
import sympy as sp
from psiop import * 



# =====================================================================
# DEMONSTRATION WITH QUALITY DIAGNOSTICS
# =====================================================================
if __name__ == "__main__":
    x, xi = sp.symbols("x xi", real=True)
    p_1d = sp.sqrt(1 + 0.2 * sp.cos(x) + xi**2)
    bounds_1d = {x: (-3.0, 3.0), xi: (-10.0, 10.0)}

    pairs_1d, metrics_1d = factorize_symbolic(
        p_1d, [x], [xi], bounds_1d, degree=6, tol=1e-4
    )

    print("=== Decomposition Quality Diagnostics ===")
    print(f"Modes Extracted     : {len(pairs_1d)}")
    print(f"SVD Energy Retained : {metrics_1d['svd_energy_retained_pct']:.4f}%")
    print(f"Relative L2 Error   : {metrics_1d['rel_l2_error']:.6e}")
    print(f"Max Absolute Error  : {metrics_1d['max_abs_error']:.6e}")
    print(f"Mean Absolute Error : {metrics_1d['mean_abs_error']:.6e}")

In [ ]:
# =====================================================================
# BENCHMARK SUITE: 3 1D EXAMPLES AND 3 2D EXAMPLES
# =====================================================================

def run_suite():
    # Define symbolic variables
    x, xi = sp.symbols("x xi", real=True)
    x1, x2, xi1, xi2 = sp.symbols("x1 x2 xi1 xi2", real=True)

    # -----------------------------------------------------------------
    # 1D BENCHMARKS (1 Spatial x, 1 Spectral xi)
    # -----------------------------------------------------------------
    benchmarks_1d = [
        (
            "1D Benchmark 1: Gaussian Non-linear Coupling",
            sp.exp(-0.5 * (x**2 + xi**2 + x * xi)),
            [x],
            [xi],
            {x: (-2.0, 2.0), xi: (-2.0, 2.0)},
            6,
            1e-4,
        ),
        (
            "1D Benchmark 2: Rational Kernel (Advection-Diffusion)",
            1 / (1 + 0.3 * x * sp.sin(xi)),
            [x],
            [xi],
            {x: (-1.0, 1.0), xi: (-np.pi, np.pi)},
            6,
            1e-4,
        ),
        (
            "1D Benchmark 3: Trigonometric-Exponential Field",
            sp.sin(sp.pi * x) * sp.exp(0.3 * xi * sp.cos(x)),
            [x],
            [xi],
            {x: (0.0, 1.0), xi: (-1.0, 1.0)},
            6,
            1e-4,
        ),
    ]

    # -----------------------------------------------------------------
    # 2D BENCHMARKS (2 Spatial x1, x2 | 2 Spectral xi1, xi2)
    # -----------------------------------------------------------------
    benchmarks_2d = [
        (
            "2D Benchmark 1: Regularized Green's Function Kernel",
            1 / sp.sqrt((x1 - xi1)**2 + (x2 - xi2)**2 + 0.5),
            [x1, x2],
            [xi1, xi2],
            {x1: (-1.0, 1.0), x2: (-1.0, 1.0), xi1: (-1.0, 1.0), xi2: (-1.0, 1.0)},
            5,
            1e-3,
        ),
        (
            "2D Benchmark 2: Coupled Multi-Frequency Wave",
            sp.cos(x1 * xi1 + x2 * xi2),
            [x1, x2],
            [xi1, xi2],
            {x1: (0.0, np.pi / 2), x2: (0.0, np.pi / 2), xi1: (0.5, 1.5), xi2: (0.5, 1.5)},
            5,
            1e-3,
        ),
        (
            "2D Benchmark 3: Non-separable Polynomial Field",
            (1 + 0.2 * x1 * x2) * (1 + xi1**2 + xi2)**2 + 0.1 * x1 * xi2,
            [x1, x2],
            [xi1, xi2],
            {x1: (-1.0, 1.0), x2: (-1.0, 1.0), xi1: (-1.0, 1.0), xi2: (-1.0, 1.0)},
            4,
            1e-5,
        ),
    ]

    # Execute and output diagnostic reports
    for name, expr, x_syms, xi_syms, bounds, deg, tol in benchmarks_1d + benchmarks_2d:
        pairs, metrics = factorize_symbolic(
            expr, x_syms, xi_syms, bounds, degree=deg, tol=tol
        )
        print(f"=== {name} ===")
        print(f"Expression           : {expr}")
        print(f"Spatial Dim (d_x)    : {len(x_syms)} | Spectral Dim (d_xi): {len(xi_syms)}")
        print(f"Modes Extracted      : {len(pairs)}")
        print(f"SVD Energy Retained  : {metrics['svd_energy_retained_pct']:.4f}%")
        print(f"Relative L2 Error    : {metrics['rel_l2_error']:.6e}")
        print(f"Max Absolute Error   : {metrics['max_abs_error']:.6e}")
        print(f"Mean Absolute Error  : {metrics['mean_abs_error']:.6e}\n")

if __name__ == "__main__":
    run_suite()

In [ ]:
def run_new_benchmark_suite(
    num_samples=10000,
    seed=42,
    digits=5,
    include_extended=True,
    raise_on_failure=False,
):
    """
    Benchmark/test suite for the newer factorize_symbolic implementation.

    Parameters
    ----------
    num_samples : int
        Monte Carlo samples used for quality diagnostics.
    seed : int
        RNG seed.
    digits : int
        Default SymPy floating coefficient precision.
    include_extended : bool
        If True, run additional cases for complex/constant/zero behavior.
        If False, run only the original-style 1D/2D cases.
    raise_on_failure : bool
        If True, raise AssertionError if any case fails basic sanity checks.
        Useful for pytest/CI.
    """
    x, xi = sp.symbols("x xi", real=True)
    x1, x2, xi1, xi2 = sp.symbols("x1 x2 xi1 xi2", real=True)

    # =================================================================
    # Original-style benchmark cases
    # =================================================================
    cases = [
        dict(
            name="1D Benchmark 1: Gaussian Non-linear Coupling",
            expr=sp.exp(-0.5 * (x**2 + xi**2 + x * xi)),
            x_syms=[x],
            xi_syms=[xi],
            bounds={x: (-2.0, 2.0), xi: (-2.0, 2.0)},
            degree=6,
            tol=1e-4,
        ),
        dict(
            name="1D Benchmark 2: Rational Kernel (Advection-Diffusion)",
            expr=1 / (1 + 0.3 * x * sp.sin(xi)),
            x_syms=[x],
            xi_syms=[xi],
            bounds={x: (-1.0, 1.0), xi: (-np.pi, np.pi)},
            degree=6,
            tol=1e-4,
        ),
        dict(
            name="1D Benchmark 3: Trigonometric-Exponential Field",
            expr=sp.sin(sp.pi * x) * sp.exp(0.3 * xi * sp.cos(x)),
            x_syms=[x],
            xi_syms=[xi],
            bounds={x: (0.0, 1.0), xi: (-1.0, 1.0)},
            degree=6,
            tol=1e-4,
        ),
        dict(
            name="2D Benchmark 1: Regularized Green's Function Kernel",
            expr=1 / sp.sqrt((x1 - xi1)**2 + (x2 - xi2)**2 + 0.5),
            x_syms=[x1, x2],
            xi_syms=[xi1, xi2],
            bounds={
                x1: (-1.0, 1.0),
                x2: (-1.0, 1.0),
                xi1: (-1.0, 1.0),
                xi2: (-1.0, 1.0),
            },
            degree=5,
            tol=1e-3,
        ),
        dict(
            name="2D Benchmark 2: Coupled Multi-Frequency Wave",
            expr=sp.cos(x1 * xi1 + x2 * xi2),
            x_syms=[x1, x2],
            xi_syms=[xi1, xi2],
            bounds={
                x1: (0.0, np.pi / 2),
                x2: (0.0, np.pi / 2),
                xi1: (0.5, 1.5),
                xi2: (0.5, 1.5),
            },
            degree=5,
            tol=1e-3,
        ),
        dict(
            name="2D Benchmark 3: Non-separable Polynomial Field",
            expr=(1 + 0.2 * x1 * x2) * (1 + xi1**2 + xi2)**2 + 0.1 * x1 * xi2,
            x_syms=[x1, x2],
            xi_syms=[xi1, xi2],
            bounds={
                x1: (-1.0, 1.0),
                x2: (-1.0, 1.0),
                xi1: (-1.0, 1.0),
                xi2: (-1.0, 1.0),
            },
            degree=4,
            tol=1e-5,
        ),
    ]

    # =================================================================
    # Extended cases for the newer implementation
    # =================================================================
    if include_extended:
        cases += [
            dict(
                name="Extended 1D: Complex Oscillatory Coupling",
                expr=sp.exp(sp.I * x * xi) + 0.25 * sp.I * sp.sin(x) * sp.cos(xi),
                x_syms=[x],
                xi_syms=[xi],
                bounds={x: (-1.0, 1.0), xi: (-np.pi, np.pi)},
                degree=6,
                tol=1e-4,
            ),
            dict(
                name="Extended 2D: Complex Rational-Wave Kernel",
                expr=(
                    sp.cos(x1 * xi1) + sp.I * sp.sin(x2 * xi2)
                ) / (2.0 + 0.2 * (x1 + xi2)),
                x_syms=[x1, x2],
                xi_syms=[xi1, xi2],
                bounds={
                    x1: (-1.0, 1.0),
                    x2: (-1.0, 1.0),
                    xi1: (-1.0, 1.0),
                    xi2: (-1.0, 1.0),
                },
                degree=4,
                tol=1e-3,
            ),
            dict(
                name="Extended: Exactly Separable Polynomial",
                expr=(1 + x + x**2) * (1 - 0.5 * xi + xi**2),
                x_syms=[x],
                xi_syms=[xi],
                bounds={x: (-1.0, 1.0), xi: (-1.0, 1.0)},
                degree=3,
                tol=1e-10,
                digits=15,
                max_modes=2,
                max_rel_l2_error=1e-8,
            ),
            dict(
                name="Extended: Constant Complex Expression",
                expr=2 + 3 * sp.I,
                x_syms=[x],
                xi_syms=[xi],
                bounds={x: (-1.0, 1.0), xi: (-1.0, 1.0)},
                degree=3,
                tol=1e-10,
                digits=15,
                max_modes=2,
                max_rel_l2_error=1e-8,
            ),
            dict(
                name="Extended: Zero Expression",
                expr=sp.S.Zero,
                x_syms=[x],
                xi_syms=[xi],
                bounds={x: (-1.0, 1.0), xi: (-1.0, 1.0)},
                degree=3,
                tol=1e-10,
                allow_empty=True,
                expected_modes=0,
                max_rel_l2_error=1e-12,
            ),
        ]

    results = []

    for case in cases:
        name = case["name"]
        expr = case["expr"]
        x_syms = list(case["x_syms"])
        xi_syms = list(case["xi_syms"])
        bounds = case["bounds"]

        pairs, metrics = factorize_symbolic(
            expr,
            x_syms,
            xi_syms,
            bounds,
            degree=case.get("degree", 5),
            tol=case.get("tol", 1e-5),
            num_samples=case.get("num_samples", num_samples),
            seed=case.get("seed", seed),
            digits=case.get("digits", digits),
        )

        issues = []

        # ------------------------------------------------------------
        # Basic sanity checks
        # ------------------------------------------------------------
        finite_ok = (
            np.isfinite(metrics.get("rel_l2_error", np.inf))
            and np.isfinite(metrics.get("max_abs_error", np.inf))
            and np.isfinite(metrics.get("mean_abs_error", np.inf))
            and np.isfinite(metrics.get("svd_energy_retained_pct", np.inf))
        )

        sv = np.asarray(metrics.get("singular_values", []))
        if sv.size > 0:
            finite_ok = finite_ok and bool(np.all(np.isfinite(sv)))

        if not finite_ok:
            issues.append("non-finite metric detected")

        if len(pairs) == 0 and not case.get("allow_empty", False):
            issues.append("no modes extracted")

        if "expected_modes" in case and len(pairs) != case["expected_modes"]:
            issues.append(
                f"expected {case['expected_modes']} modes, got {len(pairs)}"
            )

        if "max_modes" in case and len(pairs) > case["max_modes"]:
            issues.append(
                f"expected at most {case['max_modes']} modes, got {len(pairs)}"
            )

        if "max_rel_l2_error" in case:
            if metrics["rel_l2_error"] > case["max_rel_l2_error"]:
                issues.append(
                    "relative L2 error "
                    f"{metrics['rel_l2_error']:.6e} exceeds bound "
                    f"{case['max_rel_l2_error']:.6e}"
                )

        if "max_max_abs_error" in case:
            if metrics["max_abs_error"] > case["max_max_abs_error"]:
                issues.append(
                    "max absolute error "
                    f"{metrics['max_abs_error']:.6e} exceeds bound "
                    f"{case['max_max_abs_error']:.6e}"
                )

        results.append(
            dict(
                name=name,
                expr=expr,
                x_syms=x_syms,
                xi_syms=xi_syms,
                bounds=bounds,
                pairs=pairs,
                metrics=metrics,
                issues=issues,
            )
        )

        # ------------------------------------------------------------
        # Report
        # ------------------------------------------------------------
        print("=" * 70)
        print(f"=== {name} ===")
        print(f"Expression           : {expr}")
        print(
            f"Spatial Dim (d_x)    : {len(x_syms)} | "
            f"Spectral Dim (d_xi)  : {len(xi_syms)}"
        )
        print(f"Modes Extracted      : {len(pairs)}")
        print(
            "SVD Energy Retained  : "
            f"{metrics['svd_energy_retained_pct']:.6f}%"
        )
        print(f"Relative L2 Error    : {metrics['rel_l2_error']:.6e}")
        print(f"Max Absolute Error   : {metrics['max_abs_error']:.6e}")
        print(f"Mean Absolute Error  : {metrics['mean_abs_error']:.6e}")

        if sv.size > 0:
            nshow = min(5, sv.size)
            print(
                "Leading singulars    : "
                f"{np.array2string(sv[:nshow], precision=6)}"
            )

        if issues:
            print("Issues               :")
            for issue in issues:
                print(f"  - {issue}")
        else:
            print("Status               : OK")

        print()

    # =================================================================
    # Summary
    # =================================================================
    failed = [r for r in results if r["issues"]]

    print("=" * 70)
    print(
        f"Summary: {len(results) - len(failed)} / {len(results)} "
        "benchmark cases passed."
    )

    if failed:
        print("\nFailed cases:")
        for r in failed:
            print(f"  - {r['name']}")
            for issue in r["issues"]:
                print(f"      * {issue}")

    if failed and raise_on_failure:
        msg = "\n".join(
            [f"{r['name']}: {'; '.join(r['issues'])}" for r in failed]
        )
        raise AssertionError(msg)

    return results


# =====================================================================
# Optional pytest entry point
# =====================================================================
def test_new_factorization_suite():
    """
    Lightweight pytest wrapper.

    Uses fewer Monte Carlo samples so it is more suitable for CI.
    The main cost is usually the tensor-grid construction/SVD, not the
    Monte Carlo evaluation, so you can further reduce num_samples if needed.
    """
    run_new_benchmark_suite(
        num_samples=2000,
        raise_on_failure=True,
        include_extended=True,
    )


if __name__ == "__main__":
    run_new_benchmark_suite(
        num_samples=10000,
        raise_on_failure=False,
        include_extended=True,
    )